# Source

> Source code for **y**et**a**nother**s**olve**i**t

`JupyterChat` is a thin facade that wires the three parts of yasi together:

- `yasi.dialog` — pure notebook-JSON → messages logic
- `yasi.client` — the OpenAI-compatible API wrapper
- `yasi.ui` — the ipylab panel and toolbar

Its public API is unchanged, so `from yasi.core import JupyterChat` keeps working as before.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from ipylab import JupyterFrontEnd
import json

from yasi.dialog import tag_in_cell, extract_dialog
from yasi.client import ChatClient
from yasi.ui import YasiUI
from yasi.magics import load_prompt_magic

_all_ = ['tag_in_cell']

class JupyterChat:
    """Integrates a chatbot into JupyterLab, allowing users to interact with an OpenAI model directly within notebooks."""
    def __init__(self,
                 api_key : str=None,  # api key for the openai api
                 openai_base_url : str =None, # base url of the openai api
                 model : str =None, # model id for the openai api
                 tag_system : str ='#| chat_system', # tag for system chat markdown cells
                 tag_user : str ='#| chat_user', # tag for user chat markdown cells
                 tag_assistant : str ='#| chat_assistant', # tag for assistant chat markdown cells
                 all_cells : bool =True, # send untagged markdown/code cells (incl. outputs) as user messages
                 max_output_len : int =2000 # maximum characters of rendered output per code cell
                ):
        self.chat_client = ChatClient(api_key=api_key, openai_base_url=openai_base_url)
        self.client = self.chat_client.client  # kept for backward compatibility
        self.app = JupyterFrontEnd()
        self.get_models()
        self.model = model
        self.latest_response = None
        self.tag_system =  tag_system
        self.tag_user = tag_user
        self.tag_assistant = tag_assistant

        self.all_cells = all_cells
        self.max_output_len = max_output_len

        self.max_tokens = None
        self.temperature = None
        self.presence_penalty = None

        self.ui = YasiUI(app=self.app,
                         models=self.models,
                         on_send_dialog=self.send_dialog,
                         on_model_change=self._on_model_change,
                         on_params_change=self._on_params_change,
                         tag_user=tag_user,
                         tag_assistant=tag_assistant)

        try:
            load_prompt_magic(self)
        except Exception:
            pass  # not running inside IPython

    def __getattr__(self, name):
        # keep old attribute access like `jc.wgt_dd_model` or `jc.panel` working
        if name.startswith('wgt_') or name == 'panel':
            ui = self.__dict__.get('ui')
            if ui is not None: return getattr(ui, name)
        raise AttributeError(f"'{type(self).__name__}' object has no attribute '{name}'")

    def _on_model_change(self, model_id):
        self.model = model_id

    def _on_params_change(self, max_tokens, temperature, presence_penalty):
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.presence_penalty = presence_penalty

    def get_models(self):
        """Create list of model objects from the Openai client"""
        self.models = self.chat_client.get_models()

    def add_message_cell(self, id):
        self.ui.add_message_cell(id)

    def create_new_markdown_cell(self,
                                 content: str # Markdown content
                                ):
        """Adds a new markdown cell with the given content below"""
        self.ui.create_new_markdown_cell(content)

    def send_chat_completions(self,
                              messages: list # A list of messages comprising the conversation so far.
                             ):
        """Send messages a.k.a dialog to the Openai chat completions API"""
        response_text = self.chat_client.chat(messages,
                                              model=self.model,
                                              max_tokens=self.max_tokens,
                                              temperature=self.temperature,
                                              presence_penalty=self.presence_penalty)
        self.latest_response = self.chat_client.latest_response

        # Insert a new cell below with the assistant's response, tagged so
        # the next extraction sends it with the assistant role
        self.create_new_markdown_cell(f"{self.tag_assistant}\n\n{response_text.strip()}")

    def send_query(self, user_prompt: str):
        """Send user prompt to chatbot and insert response as new cell"""
        self.send_chat_completions(messages=[{"role": "user", "content": user_prompt}])

    def get_current_nb(self):
        """Saves the the notebook returns the JSON content"""
        self.app.commands.execute('docmanager:save')
        fn = self.app.sessions.current_session['name']
        with open(fn, 'r') as f:
            current_notebook = json.load(f)
        return current_notebook

    def extract_notebook_dialog(self):
        """Extracts the dialog form the current notebook, and turns it into a messages list"""
        current_notebook = self.get_current_nb()
        messages, warnings = extract_dialog(current_notebook,
                                            tag_system=self.tag_system,
                                            tag_user=self.tag_user,
                                            tag_assistant=self.tag_assistant,
                                            all_cells=self.all_cells,
                                            max_output_len=self.max_output_len)
        for warning in warnings:
            self.create_new_markdown_cell(warning)
        return messages

    def send_dialog(self, id=None):
        """Sends the dialoge to the openai api and adds the response as a new cell below"""
        self.send_chat_completions(messages=self.extract_notebook_dialog())

Usage (needs a running JupyterLab frontend and an api key):

In [ ]:
#| notest
jc = JupyterChat(openai_base_url="https://openrouter.ai/api/v1", api_key=None)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()